In [1]:
import base64
import os
import uuid

from google.adk.agents import LlmAgent
from google.adk.apps.app import App, ResumabilityConfig
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.function_tool import FunctionTool
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.tool_context import ToolContext
from google.genai import types
from IPython.display import Image as IPImage
from IPython.display import display
from mcp import StdioServerParameters

In [2]:
os.environ["GOOGLE_API_KEY"] = "REDACTED_GOOGLE_API_KEY"

In [3]:
retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

mcp_server = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-everything"],
            tool_filter=["getTinyImage"],
        ),
        timeout=60,
    )
)


In [4]:
def request_image_approval(num_images: int, tool_context: ToolContext) -> dict:
    if num_images <= 1:
        return {"status": "approved", "message": "Авто-одобрение для 1 изображения."}

    if not tool_context.tool_confirmation:
        tool_context.request_confirmation(
            hint=f"Запрос на массовую генерацию: {num_images} шт. Одобрить?",
            payload={"num_images": num_images},
        )
        return {"status": "pending", "message": "Жду одобрения человека..."}

    if tool_context.tool_confirmation.confirmed:
        return {
            "status": "approved",
            "message": f"Одобрено человеком: {num_images} изображений.",
        }
    else:
        return {"status": "rejected", "message": "Отклонено человеком."}

In [ ]:
image_agent = LlmAgent(
    name="image_agent",
    model=Gemini(model="gemini-2.5-flash", retry_options=retry_config),
    instruction="""Ты ассистент генерации изображений.
    
    АЛГОРИТМ:
    1. Извлеки количество изображений (num_images) из запроса пользователя.
    2. Вызови функцию `request_image_approval` с параметром num_images.
    3. Проанализируй ответ:
       - Если status = 'approved' → НЕМЕДЛЕННО вызови `getTinyImage` ровно num_images раз (по одному разу для каждого изображения).
       - Если status = 'pending' → сообщи пользователю "Ожидаю вашего решения..."
       - Если status = 'rejected' → сообщи "Генерация отменена по вашему запросу."
    
    ВАЖНО: При status='approved' ты ДОЛЖЕН вызвать getTinyImage, не спрашивай дополнительно!
    """,
    tools=[mcp_server, FunctionTool(func=request_image_approval)],
)

image_app = App(
    name="image_generation_app",
    root_agent=image_agent,
    resumability_config=ResumabilityConfig(is_resumable=True),
)

session_service = InMemorySessionService()
image_runner = Runner(app=image_app, session_service=session_service)

/var/folders/g7/1pnn3np5387b0n86j69n94v40000gn/T/ipykernel_74289/2398110356.py:17: UserWarning: [EXPERIMENTAL] ResumabilityConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  resumability_config=ResumabilityConfig(is_resumable=True),


In [6]:
def check_for_approval(events):
    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if (
                    part.function_call
                    and part.function_call.name == "adk_request_confirmation"
                ):
                    return {
                        "approval_id": part.function_call.id,
                        "invocation_id": event.invocation_id,
                    }
    return None


def create_approval_response(approval_info, approved):
    confirmation_response = types.FunctionResponse(
        id=approval_info["approval_id"],
        name="adk_request_confirmation",
        response={"confirmed": approved},
    )
    return types.Content(
        role="user", parts=[types.Part(function_response=confirmation_response)]
    )


def print_events(events):
    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"Agent > {part.text}")
                if hasattr(part, "function_response") and part.function_response:
                    for item in part.function_response.response.get("content", []):
                        if item.get("type") == "image":
                            display(IPImage(data=base64.b64decode(item["data"])))

In [ ]:
async def run_interactive_workflow(query: str):
    print(f"\n{'=' * 60}")
    print(f"Пользователь > {query}\n")

    session_id = f"img_session_{uuid.uuid4().hex[:8]}"
    await session_service.create_session(
        app_name="image_generation_app", user_id="user", session_id=session_id
    )

    query_content = types.Content(role="user", parts=[types.Part(text=query)])
    events = []

    async for event in image_runner.run_async(
        user_id="user", session_id=session_id, new_message=query_content
    ):
        events.append(event)

    print_events(events)

    approval_info = check_for_approval(events)

    if approval_info:
        print("\n⏸ АГЕНТ ПРИОСТАНОВЛЕН. ТРЕБУЕТСЯ ВАШЕ РЕШЕНИЕ.")

        while True:
            user_input = input("Разрешить генерацию? (y/n): ").strip().lower()
            if user_input in ["y", "yes", "д", "да"]:
                is_approved = True
                break
            elif user_input in ["n", "no", "н", "нет"]:
                is_approved = False
                break
            else:
                print("Пожалуйста, введите 'y' или 'n'")
        print("\n Возобновление работы агента...\n")

        resume_events = []
        async for event in image_runner.run_async(
            user_id="user",
            session_id=session_id,
            new_message=create_approval_response(approval_info, is_approved),
            invocation_id=approval_info["invocation_id"],
        ):
            resume_events.append(event)

        print_events(resume_events)
    else:
        print("Автоматическое одобрение — изображение сгенерировано.")

    print(f"{'=' * 60}\n")

In [8]:
await run_interactive_workflow("Сгенерируй 3 картинки")


Пользователь > Сгенерируй 3 картинки



/Users/arsknz/miniforge3/envs/main_nn/lib/python3.10/site-packages/google/adk/tools/mcp_tool/mcp_tool.py:103: UserWarning: [EXPERIMENTAL] BaseAuthenticatedTool: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__(
/Users/arsknz/miniforge3/envs/main_nn/lib/python3.10/site-packages/google/adk/tools/tool_context.py:92: UserWarning: [EXPERIMENTAL] ToolConfirmation: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  ToolConfirmation(



 АГЕНТ ПРИОСТАНОВЛЕН. ТРЕБУЕТСЯ ВАШЕ РЕШЕНИЕ.

...Возобновление работы агента...



/Users/arsknz/miniforge3/envs/main_nn/lib/python3.10/site-packages/google/adk/agents/invocation_context.py:302: UserWarning: [EXPERIMENTAL] BaseAgentState: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self.agent_states[event.author] = BaseAgentState()


Agent > Вот ваши 3 изображения: Это крошечное изображение: Это крошечное изображение: Это крошечное изображение:



In [10]:
await run_interactive_workflow("Сгенерируй 1 картинку")

Error during disconnected session cleanup: Attempted to exit cancel scope in a different task than it was entered in



Пользователь > Сгенерируй 1 картинку



/Users/arsknz/miniforge3/envs/main_nn/lib/python3.10/site-packages/google/adk/tools/mcp_tool/mcp_tool.py:103: UserWarning: [EXPERIMENTAL] BaseAuthenticatedTool: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__(


Agent > Вот 1 картинка: This is a tiny image:
The image above is the MCP tiny image.


